# Agent 智能体设计模式
## Orchestrator-workers
在协调器-工作者工作流中，中央 LLM 动态分解任务，将其委托给工作者 LLM，并综合其结果。
- 由一个大模型作为管理员对复杂任务进行拆分
- 拆分成子任务以及任务说明，再分发给其它大模型进行下一步执行
- 适合于无法在事前确定子任务个数


![](https://i-blog.csdnimg.cn/direct/67d04c881e9b41c5ad8a21a06f0708cd.png)

In [1]:
from openai import OpenAI
from datetime import datetime
import json
from typing import List, Dict, Callable
import os
import re
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
load_dotenv("/Users/a1-6/Documents/projects/DL/.env")
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

llm_name = llm_name = "qwen-plus"
def call_llm(user_prompt, system_prompt=""):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user","content": user_prompt}
        ]
    response = client.chat.completions.create(
            model= llm_name,
            messages = messages)
    return(response.choices[0].message.content)

In [2]:
def extract_xml(text: str, tag: str) -> str:
    match = re.search(f'<{tag}>(.*?)</{tag}>', text, re.DOTALL)
    return match.group(1) if match else ""

In [3]:
def extract_tasks(xml_string):
    pattern = r'<task>(.*?)</task>'
    tasks = re.findall(pattern, xml_string,re.DOTALL)
    return tasks

In [4]:
class FlexibleOrchestrator:
    """Break down tasks and run them in parallel using worker LLMs."""
    def __init__(
        self,
        orchestrator_prompt: str,
        worker_prompt: str,
    ):
        """Initialize with prompt templates."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt


    def _format_prompt(self, template: str, **kwargs) :
        """Format a prompt template with variables."""
        try:
            return template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}")


    def process(self, task: str, context = None) :
        """Process task by breaking it down and running subtasks in parallel."""
        context = context or {}
        # Step 1: Get orchestrator response
        orchestrator_input = self._format_prompt(
            self.orchestrator_prompt,
            task=task
        )
        orchestrator_response = call_llm(orchestrator_input)
        # Parse orchestrator response
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = extract_tasks(tasks_xml)
        print("\n=== ORCHESTRATOR OUTPUT ===")
        print(f"\nANALYSIS:\n{analysis}")
        print(f"\nTASKS:\n{tasks}")
        # Step 2: Process each task
        worker_results = []

        for task_info in tasks:
            worker_input = self._format_prompt(
                self.worker_prompt,
                original_task=task,
                task_description=task_info
            )

            worker_response = call_llm(worker_input)

            result = extract_xml(worker_response, "response")
            
            worker_results.append({
                "description": task_info,
                "result": result
            })
            print(f"\n=== WORKER RESULT ({task_info}) ===\n{result}\n")


        return {
            "analysis": analysis,
            "worker_results": worker_results,
        }




In [5]:
ORCHESTRATOR_PROMPT = """
你需要分析下面的问题，并将其分拆成几个不同的立场进行多视角分析。

需要分析的问题如下：\{task\}

返回格式输出示例如下：

<analysis>
    在这里解释你对需要分析的问题的理解。
</analysis>

<tasks>
    <task>在这里输出视角1</task>
    <task>在这里输出视角2</task>
</tasks>
"""


In [6]:
WORKER_PROMPT="""
基于如下要求进行回答：
你需要分析的问题如下：\{original_task\}
你的分析视角如下：\{task_description\}

返回格式示例如下：
<response>
在这里输出你的回答内容
</response>
"""

In [ ]:
orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

In [ ]:
orchestrator.process(task="Ai时代需要学习编程")